## llamacpp_unsloth_Qwen3.5-27B-GGUF_UD-Q4_K_XL
llamacpp を使う場合は 172.17.0.1 で接続する必要がある。

172.17.0.1は、主にDocker（ドッカー）環境におけるデフォルトのブリッジネットワーク（docker0）のホスト側のIPアドレスです。

## 所感

- さすがにRAGの精度は下がるけど、精度とスループットのバランスは良いと思うし、完全にローカルで実行できるメリットもある。
- sashisuseso_GPT-OSS-Swallow-20B-SFT-v0.1-MXFP4 は速度は早いけど、コンテキストからうまく回答ができなかったので使えない。

In [34]:
import json
import time
import httpx

BASE_URL = "http://172.17.0.1:1067"
MODEL    = "unsloth/Qwen3.5-27B-GGUF"

HEADERS = {
    "Content-Type": "application/json",
    "Authorization": "Bearer no-key",
}

def wait_until_ready(timeout_sec: float = 60.0) -> None:
    deadline = time.time() + timeout_sec
    while time.time() < deadline:
        r = httpx.get(f"{BASE_URL}/health", timeout=5.0)
        if r.status_code == 200:
            return
        if r.status_code == 503:
            time.sleep(0.5)
            continue
        r.raise_for_status()
    raise TimeoutError("llama-server が ready になりませんでした")

def stream_chat(user_text: str, system_text: str = "あなたは高度なAIアシスタントです。英語で考えて日本語で回答してください。") -> None:
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_text},
            {"role": "user", "content": user_text},
        ],
        "temperature": 0.7,
        "top_p": 0.8,
        "top_k": 20,
        "min_p": 0.0,
        "stream": True,
    }

    with httpx.Client(base_url=BASE_URL, headers=HEADERS, timeout=None) as client:
        # httpx は stream=True ではなく client.stream(...) を使う :contentReference[oaicite:5]{index=5}
        with client.stream("POST", "/v1/chat/completions", json=payload) as resp:
            resp.raise_for_status()

            for line in resp.iter_lines():
                if not line:
                    continue

                # SSE: "data: ...."
                if not line.startswith("data:"):
                    continue

                data_str = line[len("data:"):].strip()
                if data_str == "[DONE]":
                    break

                chunk = json.loads(data_str)

                # OpenAI 互換: choices[0].delta.content に増分が入ることが多い
                choice0 = (chunk.get("choices") or [{}])[0]
                delta = choice0.get("delta") or {}
                text = delta.get("content")
                if text:
                    print(text, end="", flush=True)

    print()  # 改行

if __name__ == "__main__":
    wait_until_ready()
    stream_chat("RustでHTTPサーバーを立てる最小例を説明して。")

RustでHTTPサーバーを構築する最小限の例を説明します。

Rustの標準ライブラリ（標準庫）には、HTTPサーバーを構築するための完全な機能は含まれていないため、通常は**`actix-web`**、**`axum`**、**`warp`**などのフレームワークを使用します。その中でも、現在最も人気でモダンな**`actix-web`**を使用した例を紹介します。

この例では、以下の手順で「Hello, World!」を返すサーバーを作成します。

1.  新しいプロジェクトの作成
2.  `Cargo.toml`への依存関係の追加
3.  `main.rs`へのコード記述
4.  ビルドと実行

### 1. プロジェクトの作成

ターミナル（コマンドプロンプト）で以下を実行して、新しいRustプロジェクトを作成します。

```bash
cargo new hello_server
cd hello_server
```

### 2. 依存関係の追加 (`Cargo.toml`)

`Cargo.toml`ファイルを開き、`[dependencies]`セクションに`actix-web`と`tokio`を追加します。`tokio`は非同期実行環境（ランタイム）で、`actix-web`が動作するために必須です。

```toml
[package]
name = "hello_server"
version = "0.1.0"
edition = "2021"

[dependencies]
actix-web = "4"
tokio = { version = "1", features = ["full"] }
```

### 3. コードの記述 (`src/main.rs`)

`src/main.rs`を以下のように書き換えます。このコードは、`localhost:8080`でサーバーを起動し、ルートパス`/`にアクセスすると"Hello, World!"を返すようにしています。

```rust
use actix_web::{web, App, HttpResponse, HttpServer, Responder};

// ハンドラ関数: リクエストを受け取り、レスポンスを返す
async fn hello() -> impl Res

# MiniRAG を postgres で実行する

- 精度は安定していないところもあるし、まだバグが潜んでいる可能性もあるが、一旦このレベルで OK

In [6]:
# 必要なライブラリのインポート
import os
import tempfile
from minirag import MiniRAG, QueryParam
from minirag.llm.hf import (
    hf_model_complete,
    hf_embed,
)
# from minirag.llm.openai import openrouter_openai_complete
# from minirag.llm.openai import openai_complete_if_cache
from minirag.utils import EmbeddingFunc
from minirag.utils import (
    wrap_embedding_func_with_attrs,
    locate_json_string_body_from_string,
    safe_unicode_decode,
    logger,
)

import asyncpg
from psycopg_pool import AsyncConnectionPool
from minirag.kg.postgres_impl import PostgreSQLDB
from minirag.kg.postgres_impl import PGKVStorage
from minirag.kg.postgres_impl import PGVectorStorage
from minirag.kg.postgres_impl import PGGraphStorage
from minirag.kg.postgres_impl import PGDocStatusStorage

from transformers import AutoModel, AutoTokenizer
from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_exception
from openai import RateLimitError, APIStatusError # openaiライブラリが投げる例外をインポート
import asyncio
import warnings
warnings.filterwarnings('ignore')

### データベース接続テスト

In [7]:
import psycopg
from psycopg.rows import dict_row

def get_conn():
    url = os.getenv("DATABASE_URL")
    if url:
        return psycopg.connect(url, row_factory=dict_row)
    # fallback: assemble from separate vars
    dsn = (
        f"host={os.getenv('PGHOST','db')} "
        f"port={os.getenv('PGPORT','5432')} "
        f"dbname={os.getenv('PGDATABASE','postgres')} "
        f"user={os.getenv('POSTGRES_USER','postgres')} "
        f"password={os.getenv('POSTGRES_PASSWORD')}"
    )
    return psycopg.connect(dsn, row_factory=dict_row)

with get_conn() as conn, conn.cursor() as cur:
    cur.execute("SELECT version();")
    print(cur.fetchone())

{'version': 'PostgreSQL 16.11 (Debian 16.11-1.pgdg12+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit'}


### ハイブリッドクエリのテスト（任意）
- 前提: 先に init_db.sh が実行されテーブルが作成されていること

In [8]:
def get_conn():
    """
    環境変数からデータベース接続情報を取得し、接続オブジェクトを返す。
    DATABASE_URLが設定されていればそれを使用し、なければ個別の変数を組み立てる。
    結果は辞書形式で返されるように設定済み。
    """
    url = os.getenv("DATABASE_URL")
    if url:
        # DATABASE_URLが設定されている場合
        return psycopg.connect(url, row_factory=dict_row)
    
    # DATABASE_URLがない場合、個別の環境変数からDSNを組み立てる
    # PGPASSWORDが設定されていないと接続に失敗するため、チェックを追加するとより親切
    password = os.getenv('PGPASSWORD')
    if not password:
        raise ValueError("環境変数 PGPASSWORD が設定されていません。")
        
    dsn = (
        f"host={os.getenv('PGHOST', 'localhost')} "
        f"port={os.getenv('PGPORT', '5432')} "
        f"dbname={os.getenv('PGDATABASE', 'postgres')} "
        f"user={os.getenv('POSTGRES_USER', 'postgres')} "
        f"password={password}"
    )
    return psycopg.connect(dsn, row_factory=dict_row)

def find_recommended_products_for_alice(_results):
    """
    指定されたSQLクエリを実行し、'Alice'が好む商品とベクトル的に類似した商品を取得する。
    """
    # 実行したいSQLクエリを三重クォートで定義
    # これにより、複数行のクエリを読みやすく記述できる
    sql_query = """
    SELECT
        p.id,
        p.name,
        p.embedding
    FROM
        public.products AS p
    JOIN
        -- cypher関数でグラフクエリを実行し、結果をテーブルのように扱う
        cypher('my_minirag_graph', $$
            MATCH (u:User {name: 'Alice'})-[:LIKES]->(prod:Product)
            RETURN prod.product_id
        $$) AS liked(product_id agtype)
    ON
        -- グラフクエリの結果(agtype)を整数にキャストしてproductsテーブルのIDと結合
        p.id = (liked.product_id)::INTEGER
    ORDER BY
        -- pgvectorの<->演算子で、指定ベクトルとのコサイン距離が近い順に並び替え
        p.embedding <=> '[0.1, 0.1, 0.2]';
    """

    try:
        # with文で接続とカーソルを管理し、処理終了後に自動でクローズする
        with get_conn() as conn, conn.cursor() as cur:
            print("データベースに接続し、クエリを実行します...")
            
            # クエリの実行
            cur.execute(sql_query)
            
            # 全ての実行結果を取得 (fetchall)
            # 1件だけなら fetchone(), 複数件なら fetchall() を使う
            results = cur.fetchall()
            
            print("\n--- クエリ実行結果 ---")
            if results:
                # 取得した結果を1行ずつ表示
                for row in results:
                    print(row)
                    _results.append(row)
            else:
                print("条件に一致する結果は見つかりませんでした。")
        return _results

    except psycopg.Error as e:
        print(f"データベースエラーが発生しました: {e}")
    except ValueError as e:
        print(f"設定エラーが発生しました: {e}")

query_results = []
query_results = find_recommended_products_for_alice(query_results)

データベースに接続し、クエリを実行します...

--- クエリ実行結果 ---
{'id': 1, 'name': '商品A', 'embedding': '[0.1,0.2,0.3]'}
{'id': 2, 'name': '商品B', 'embedding': '[0.2,0.1,0.9]'}


### マークダウンに整形

In [9]:
def convert_to_markdown(data: list[dict]) -> str:
    """
    辞書のリストをマークダウンのテーブル形式に変換します。

    Args:
        data: 辞書のリスト。各辞書がテーブルの1行に対応します。

    Returns:
        マークダウン形式のテーブル文字列。
    """
    if not data:
        return "データがありません。"

    # ヘッダーの作成 (最初のデータのキーから取得)
    headers = data[0].keys()
    header_line = "| " + " | ".join(headers) + " |"

    # 区切り線の作成
    separator_line = "| " + " | ".join(["---"] * len(headers)) + " |"

    # データ行の作成
    data_lines = []
    for row in data:
        # 各値を文字列に変換して結合
        values = [str(row.get(h, "")) for h in headers]
        data_lines.append("| " + " | ".join(values) + " |")

    # 全ての行を結合して返す
    return "\n".join([header_line, separator_line] + data_lines)

# --- 変換を実行して結果を表示 ---
markdown_table = convert_to_markdown(query_results)
print(markdown_table)

| id | name | embedding |
| --- | --- | --- |
| 1 | 商品A | [0.1,0.2,0.3] |
| 2 | 商品B | [0.2,0.1,0.9] |


In [10]:
#  id | name  |   embedding   
# ----+-------+---------------
#   1 | 商品A | [0.1,0.2,0.3]
#   2 | 商品B | [0.2,0.1,0.9]

# 上記の結果になればOK

In [11]:
# API KEY が見えないようにコメントアウト

# print(os.getenv("OPENROUTER_API_KEY"))
# print(os.getenv("OPENAI_API_KEY"))
print("Openrouter APIキーが設定されました")
print()

HF_TOKEN = os.getenv("HF_TOKEN")
# print(HF_TOKEN)
print("HuggingFace TOKEN が設定されました")

Openrouter APIキーが設定されました

HuggingFace TOKEN が設定されました


In [12]:
use_japanese_embedding = False

if use_japanese_embedding:
    # MINI モードでの回答ができなくなったので、逆に精度が落ちたかも。
    EMBEDDING_MODEL = "hotchpotch/static-embedding-japanese"
    EMBEDDING_DIM   = 1024
    TOKENIZER_MODEL = "hotchpotch/xlm-roberta-japanese-tokenizer"
    print("-------------- static-embedding-japanese を使用します --------------")
else:
    # 埋め込みモデルの設定
    EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
    EMBEDDING_DIM   = 384
    print("-------------- all-MiniLM-L6-v2 を使用します --------------")


# LLMの設定
# LLM_MODEL = "Qwen/Qwen3-1.7B"  # または "Qwen/Qwen3-4B", "Qwen/Qwen3-1.7B" など
# LLM_MODEL = "jaeyong2/Qwen2.5-3B-Instruct-Ja-SFT"
# LLM_MODEL = "deepseek/deepseek-chat-v3-0324:free"
# LLM_MODEL = "deepseek/deepseek-v3.2-speciale"    # 一生進まないのでこれは使わない。思考モードが悪さしている？
# LLM_MODEL = "qwen/qwen3-235b-a22b-2507"          # これは良い
# LLM_MODEL = "qwen/qwen3-30b-a3b-thinking-2507"     # 30B でも十分だった
# LLM_MODEL = "gpt-oss:20b"
LLM_MODEL = "unsloth/Qwen3.5-27B-GGUF"



# 作業ディレクトリの作成
WORKING_DIR = "/tmp/minirag_demo"
os.makedirs(WORKING_DIR, exist_ok=True)

print(f"作業ディレクトリ: {WORKING_DIR}")


# DATA_PATH = args.datapath
# QUERY_PATH = args.querypath
# OUTPUT_PATH = args.outputpath
# print("USING LLM:", LLM_MODEL)
# print("USING WORKING DIR:", WORKING_DIR)

-------------- all-MiniLM-L6-v2 を使用します --------------
作業ディレクトリ: /tmp/minirag_demo


In [13]:
if use_japanese_embedding:

    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(EMBEDDING_MODEL, device="cpu", token=HF_TOKEN)
    
    query = "美味しいラーメン屋に行きたい"
    docs = [
        "素敵なカフェが近所にあるよ。落ち着いた雰囲気でゆっくりできるし、窓際の席からは公園の景色も見えるんだ。",
        "新鮮な魚介を提供する店です。地元の漁師から直接仕入れているので鮮度は抜群ですし、料理人の腕も確かです。",
        "あそこは行きにくいけど、隠れた豚骨の名店だよ。スープが最高だし、麺の硬さも好み。",
        "おすすめの中華そばの店を教えてあげる。とりわけチャーシューが手作りで柔らかくてジューシーなんだ。",
    ]
    
    embeddings = model.encode([query] + docs)
    print(embeddings.shape)
    similarities = model.similarity(embeddings[0], embeddings[1:])
    for i, similarity in enumerate(similarities[0].tolist()):
        print(f"{similarity:.04f}: {docs[i]}")

else:
    
    tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL, token=HF_TOKEN)
    model = AutoModel.from_pretrained(EMBEDDING_MODEL, token=HF_TOKEN)


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## transformer の API で使えるように変換する

In [14]:
if use_japanese_embedding:
    
    import torch
    from torch import nn
    from transformers import PreTrainedModel, PretrainedConfig
    from transformers.modeling_outputs import BaseModelOutputWithPoolingAndCrossAttentions
    
    
    class StaticEmbeddingConfig(PretrainedConfig):
        model_type = "static-embedding"
    
        def __init__(self, vocab_size=32768, hidden_size=1024, pad_token_id=0, **kwargs):
            super().__init__(pad_token_id=pad_token_id, **kwargs)
            self.vocab_size = vocab_size
            self.hidden_size = hidden_size
    
    
    class StaticEmbeddingModel(PreTrainedModel):
        config_class = StaticEmbeddingConfig
    
        def __init__(self, config: StaticEmbeddingConfig):
            super().__init__(config)
            # ★ EmbeddingBag そのものでも OK ですが、
            #   シーケンス長をそろえて attention_mask で平均を取る方が扱いやすいので nn.Embedding に変更
            self.embedding = nn.Embedding(
                num_embeddings=config.vocab_size,
                embedding_dim=config.hidden_size,
                padding_idx=config.pad_token_id,
            )
            self.post_init()  # transformers の重み初期化
    
        def forward(self, input_ids, attention_mask=None, **kwargs):
            """
            - input_ids      : (batch, seq_len)
            - attention_mask : (batch, seq_len) — 0 は padding
            戻り値は Transformers 共通の BaseModelOutputWithPoolingAndCrossAttentions
            """
            if attention_mask is None:
                attention_mask = (input_ids != self.config.pad_token_id).int()
    
            token_embs = self.embedding(input_ids)                       # (B, L, H)
            # マスク付き平均プール
            masked_embs = token_embs * attention_mask.unsqueeze(-1)      # (B, L, H)
            lengths = attention_mask.sum(dim=1, keepdim=True).clamp(min=1e-8)  # (B, 1)
            sentence_emb = masked_embs.sum(dim=1) / lengths              # (B, H)
    
            return BaseModelOutputWithPoolingAndCrossAttentions(
                last_hidden_state=token_embs,  # ここでは token レベルをそのまま
                pooler_output=sentence_emb,    # 文ベクトル
                attentions=None,
                cross_attentions=None,
            )

In [15]:
if use_japanese_embedding:
    """
    SentenceTransformer 版 (hotchpotch/static-embedding-japanese) から
    StaticEmbeddingModel へ重みをコピーして保存するスクリプト
    """
    
    SRC = "hotchpotch/static-embedding-japanese"   # オリジナル
    DST = "./static-embedding-transformers"        # 保存先
    
    # ① SentenceTransformer を読み込む
    st_model = SentenceTransformer(SRC)
    embedding_weight = st_model[0].embedding.weight.data   # nn.EmbeddingBag の重みを取得
    
    # ② Config → Model を作成
    config = StaticEmbeddingConfig(
        vocab_size=embedding_weight.size(0),
        hidden_size=embedding_weight.size(1),
        pad_token_id=0,           # トークナイザの <pad> が id=0
    )
    model = StaticEmbeddingModel(config)
    
    # ③ 重みコピー
    with torch.no_grad():
        model.embedding.weight.copy_(embedding_weight)
    
    # ④ save_pretrained で書き出し
    model.save_pretrained(DST)
    # st_model.tokenizer.save_pretrained(DST)   # tokenizer.json なども一緒に保存
    
    print(f"✅ 変換完了 — 保存先: {DST}")

In [16]:
if use_japanese_embedding:
    
    MODEL_DIR = "./static-embedding-transformers"
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL)
    model     = StaticEmbeddingModel.from_pretrained(MODEL_DIR)
    
    sentences = [
        "美味しいラーメン屋に行きたい",
        "あそこは行きにくいけど、隠れた豚骨の名店だよ。スープが最高だし、麺の硬さも好み。",
    ]
    
    inputs = tokenizer(
        sentences,
        return_tensors="pt",
        padding=True,
        truncation=True,
        add_special_tokens=False,   # 元モデルは special tokens なし
    )
    
    with torch.no_grad():
        outputs = model(**inputs)
        vecs = outputs.pooler_output     # (batch, hidden_size)
    
    print("shape:", vecs.shape)          # torch.Size([2, 1024])
    similarity = torch.nn.functional.cosine_similarity(vecs[0], vecs[1], dim=0)
    print("cosine:", similarity.item())

In [ ]:
import sys
from typing import List, Union

if sys.version_info < (3, 9):
    from typing import AsyncIterator
else:
    from collections.abc import AsyncIterator

from openai import (
    AsyncOpenAI,
    APIConnectionError,
    RateLimitError,
)


async def openai_complete_if_cache(
    model,
    prompt,
    system_prompt=None,
    history_messages=None,
    base_url=None,
    api_key=None,
    **kwargs,
) -> Union[str, AsyncIterator[str]]:
    """
    llama-server(OpenAI互換)向けに接続・JSON(response_format)・min_p / top_k を安定化した版。

    llama-server 設定（ユーザー指定）:
      - alias: unsloth_Qwen3.5-27B-GGUF_UD-Q4_K_XL
      - host: 0.0.0.0 (待受)
      - port: 1067
      - temperature: 0.7 / top_p: 0.8 / top_k: 20 / min_p: 0.0 / stream: False
    """
    import json
    import httpx
    import os  # os.environ を使用するため
    from urllib.parse import urlparse, urlunparse

    # ---- llama-server defaults (ユーザー指定) ----
    LLAMA_SERVER_BASE_URL_DEFAULT = "http://172.17.0.1:1067/v1"
    LLAMA_SERVER_MODEL_ALIAS_DEFAULT = (
        "unsloth/Qwen3.5-27B-GGUF_UD-Q4_K_XL"
    )
    LLAMA_SERVER_DEFAULTS = {
        "temperature": 0.7,
        "top_p": 0.8,
        "top_k": 20,
        "min_p": 0.0,
        "stream": False,
    }

    def _is_official_openai(resolved_base_url: str | None) -> bool:
        if resolved_base_url is None:
            return True
        return "api.openai.com" in resolved_base_url

    def _is_pydantic_model_class(x) -> bool:
        try:
            return isinstance(x, type) and issubclass(x, BaseModel)
        except Exception:
            return False

    def _normalize_json_mode_response_format(rf):
        if isinstance(rf, dict):
            return rf
        if isinstance(rf, str) and rf in ("json", "json_object"):
            return {"type": "json_object"}
        return None

    def _augment_system_prompt_for_json(sp: str | None, rf) -> str:
        base = (sp or "").strip()
        lines: list[str] =[]

        if _is_pydantic_model_class(rf):
            try:
                schema = rf.model_json_schema()
                schema_text = json.dumps(schema, ensure_ascii=False)
                lines.append("必ずJSONだけを出力してください。前後に説明文やコードブロックは付けないでください。")
                lines.append("次のJSONスキーマに厳密に従ってください。")
                lines.append(schema_text)
            except Exception:
                lines.append("必ずJSONだけを出力してください。前後に説明文やコードブロックは付けないでください。")
        else:
            lines.append("必ずJSONオブジェクトだけを出力してください。前後に説明文やコードブロックは付けないでください。")

        addon = "\n".join(lines).strip()
        if not addon:
            return base
        if not base:
            return addon
        return f"{base}\n\n{addon}"

    def _sanitize_base_url(raw: str) -> str:
        """
        - 余計な空白除去
        - スキームが無ければ http を付与
        - host が 0.0.0.0 なら 172.17.0.1 に置換（接続先としては不適になりやすい）
        - パス末尾が /v1 になるよう補正（OpenAI SDK の base_url 期待）
        """
        s = (raw or "").strip()
        if not s:
            return LLAMA_SERVER_BASE_URL_DEFAULT
        if "://" not in s:
            s = "http://" + s

        u = urlparse(s)
        netloc = u.netloc.replace("0.0.0.0", "172.17.0.1")

        path = (u.path or "").rstrip("/")
        if path == "":
            path = "/v1"
        elif path != "/v1" and not path.endswith("/v1"):
            # 例: http://172.17.0.1:1067 -> /v1 を付ける
            path = path + "/v1"

        u2 = u._replace(netloc=netloc, path=path)
        return urlunparse(u2)

    def _resolve_base_url(explicit_base_url: str | None) -> str:
        # “こちらで設定してください”に合わせ、明示指定が無ければ llama-server 既定を使う
        if explicit_base_url is not None:
            return _sanitize_base_url(explicit_base_url)
        return _sanitize_base_url(LLAMA_SERVER_BASE_URL_DEFAULT)

    async def _preflight_healthcheck(url_v1: str) -> None:
        """
        llama-server は /health と /v1/health を提供するので、base_url(/v1) 側で叩く。
        タイムアウト：10秒(2秒は厳しい)
        """
        health_url = url_v1.rstrip("/") + "/health"
        last_error: Exception | None = None
        for _ in range(3):
            try:
                async with httpx.AsyncClient(timeout=5.0) as client:
                    r = await client.get(health_url)
                # 200 以外でも “接続できている” ことは分かるので、接続不能だけを明確に切り分けたい
                logger.debug("Healthcheck %s -> %s", health_url, r.status_code)
                return
            except Exception as e:
                last_error = e
                await asyncio.sleep(0.5)
        # preflight は補助的チェック。ここで即失敗にせず、本リクエスト側に委ねる。
        logger.warning(
            "llama-server preflight healthcheck failed (continue): base_url=%s / url=%s / reason=%r",
            url_v1,
            health_url,
            last_error,
        )

    # ---- API key ----
    if api_key:
        os.environ["OPENAI_API_KEY"] = api_key

    # ---- base_url / model ----
    resolved_base_url = _resolve_base_url(base_url)
    official_openai = _is_official_openai(resolved_base_url)

    if not model:
        model = LLAMA_SERVER_MODEL_ALIAS_DEFAULT

    # ---- client ----
    resolved_api_key = api_key or os.getenv("OPENAI_API_KEY") or "sk-local-no-key-required"
    openai_async_client = AsyncOpenAI(api_key=resolved_api_key, base_url=resolved_base_url)

    # ---- strip internal kwargs ----
    kwargs.pop("hashing_kv", None)
    kwargs.pop("keyword_extraction", None)

    # ---- llama-server sampling defaults (ユーザー指定) ----
    kwargs.setdefault("temperature", LLAMA_SERVER_DEFAULTS["temperature"])
    kwargs.setdefault("top_p", LLAMA_SERVER_DEFAULTS["top_p"])
    kwargs.setdefault("stream", LLAMA_SERVER_DEFAULTS["stream"])

    # 【修正箇所】top_k, min_p は SDK が create() 引数として受け付けないので extra_body へ移す準備
    if official_openai:
        kwargs.pop("top_k", None)
        kwargs.pop("min_p", None)
        top_k_value = None
        min_p_value = None
    else:
        top_k_value = kwargs.pop("top_k", LLAMA_SERVER_DEFAULTS["top_k"])
        min_p_value = kwargs.pop("min_p", LLAMA_SERVER_DEFAULTS["min_p"])

    # ---- messages ----
    if history_messages is None:
        history_messages =[]

    response_format = kwargs.pop("response_format", None)
    effective_system_prompt = system_prompt
    if response_format is not None:
        effective_system_prompt = _augment_system_prompt_for_json(system_prompt, response_format)

    messages =[]
    if effective_system_prompt:
        messages.append({"role": "system", "content": effective_system_prompt})
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})

    # ---- logs ----
    logger.debug("===== Query Input to LLM =====")
    logger.debug(f"Base URL: {resolved_base_url}")
    logger.debug(f"Model: {model}")
    logger.debug(f"Kwargs: { {k: v for k, v in kwargs.items() if k != 'api_key'} }")

    # ---- preflight (接続不能を早く・分かりやすく) ----
    if not official_openai:
        await _preflight_healthcheck(resolved_base_url)

    # ---- request ----
    use_stream = bool(kwargs.get("stream", False))

    # (A) OpenAI公式 + Pydantic response_format の場合は parse を試す（stream時は不可）
    if (not use_stream) and official_openai and _is_pydantic_model_class(response_format):
        try:
            parsed_resp = await openai_async_client.beta.chat.completions.parse(
                model=model,
                messages=messages,
                response_format=response_format,
                **kwargs,
            )
            msg = parsed_resp.choices[0].message if parsed_resp and parsed_resp.choices else None
            if msg is not None and getattr(msg, "parsed", None) is not None:
                out = msg.parsed.model_dump_json()
                if r"\u" in out:
                    out = safe_unicode_decode(out.encode("utf-8"))
                return out

            content = getattr(msg, "content", "") if msg is not None else ""
            if content and r"\u" in content:
                content = safe_unicode_decode(content.encode("utf-8"))
            if content:
                try:
                    return locate_json_string_body_from_string(content)
                except Exception:
                    return content
            return ""
        except Exception as e:
            logger.warning("beta.chat.completions.parse failed; fallback to create(): %s", e)

    # (B) create() 用 kwargs
    create_kwargs = dict(kwargs)

    # JSON mode（公式OpenAIのみ）
    if (not use_stream) and official_openai and response_format is not None:
        rf = _normalize_json_mode_response_format(response_format)
        if rf is not None:
            create_kwargs["response_format"] = rf
        # Pydantic は parse 失敗時にここへ来るので、create側ではプロンプト誘導で対応

    # 【修正箇所】llama-server 向け top_k, min_p を extra_body に注入
    if not official_openai:
        if top_k_value is not None or min_p_value is not None:
            extra_body = create_kwargs.pop("extra_body", None)
            if not isinstance(extra_body, dict):
                extra_body = {}
            if top_k_value is not None:
                extra_body["top_k"] = top_k_value
            if min_p_value is not None:
                extra_body["min_p"] = min_p_value
            create_kwargs["extra_body"] = extra_body

    try:
        response = await openai_async_client.chat.completions.create(
            model=model,
            messages=messages,
            **create_kwargs,
        )
    except APIConnectionError as e:
        # ここに来るのは「到達不能」系がほとんどなので、情報を付ける
        raise ConnectionError(
            "OpenAI互換サーバへの接続に失敗しました。\n"
            f"- base_url = {resolved_base_url}\n"
            "確認: `curl http://172.17.0.1:1067/v1/health` が返るかを見てください。\n"
            "注: llama-server の `--host 0.0.0.0` は待受用です。接続先は 172.17.0.1 か実IPにしてください。"
        ) from e

    # ---- streaming ----
    if hasattr(response, "__aiter__"):

        async def inner():
            async for chunk in response:
                content = chunk.choices[0].delta.content
                if content is None:
                    continue
                if r"\u" in content:
                    content = safe_unicode_decode(content.encode("utf-8"))
                yield content

        return inner()

    # ---- non-stream ----
    if not response or not hasattr(response, "choices") or not response.choices:
        logger.error("No valid choices returned. Full response: %s", response)
        return ""

    msg = response.choices[0].message
    content = getattr(msg, "content", None)
    if content is None:
        logger.error("The message content is None. Full response: %s", response)
        return ""

    if r"\u" in content:
        content = safe_unicode_decode(content.encode("utf-8"))

    # response_format 指定時は JSON 部分だけ抜き出す（llama-server等の差異吸収）
    if (not use_stream) and response_format is not None:
        try:
            return locate_json_string_body_from_string(content)
        except Exception:
            return content

    return content



# どのエラーでリトライするかを定義
# 429 (RateLimitError) や サーバー側のエラー(5xx) でリトライするのが一般的
def should_retry(e: Exception) -> bool:
    if isinstance(e, RateLimitError):
        print(f"RateLimitError発生。リトライします...: {e}")
        return True
    # 5xx系のサーバーエラーでもリトライすることが多い
    if isinstance(e, APIStatusError) and e.status_code >= 500:
        print(f"サーバーエラー( {e.status_code} )発生。リトライします...: {e}")
        return True
    if isinstance(e, APIConnectionError):
        print(f"接続エラー発生。リトライします...: {e}")
        return True
    if isinstance(e, ConnectionError):
        print(f"接続失敗発生。リトライします...: {e}")
        return True
    return False


@retry(
    wait=wait_random_exponential(multiplier=30, max=100), # 30, 32, 34, 38秒...とランダムな時間を加えて待つ（最大100秒）
    stop=stop_after_attempt(5), # 最大5回リトライする
    retry=retry_if_exception(should_retry) # 上で定義した条件の例外が発生した場合にリトライ
)
async def openai_complete(
    prompt,
    system_prompt=None,
    history_messages=None,
    keyword_extraction=False,
    **kwargs,
) -> Union[str, AsyncIterator[str]]:
    """
    - keyword_extraction=True の場合、JSONを返すように response_format を有効化
    - model は hashing_kv.global_config["llm_model_name"] を優先し、無ければ llama-server の alias を使う
    """
    if history_messages is None:
        history_messages =[]

    kw_flag = kwargs.pop("keyword_extraction", None)
    keyword_extraction = bool(keyword_extraction or kw_flag)

    if keyword_extraction and "response_format" not in kwargs:
        kwargs["response_format"] = "json"

    LLAMA_SERVER_MODEL_ALIAS_DEFAULT = (
        "unsloth/Qwen3.5-27B-GGUF_UD-Q4_K_XL"
    )
    model_name = None
    hashing_kv = kwargs.get("hashing_kv")
    try:
        model_name = hashing_kv.global_config.get("llm_model_name")
    except Exception:
        model_name = None
    if not model_name:
        model_name = LLAMA_SERVER_MODEL_ALIAS_DEFAULT

    return await openai_complete_if_cache(
        model_name,
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        **kwargs,
    )

In [25]:
!pwd

!ls

/app/docs
'!- Ollama-GPT OSS 20B はハルシネーションはあるけど使える可能性はありそう.txt'
 MiniRAG_on_Colabのサンプルコード.ipynb
 MiniRAG_on_postgres.ipynb
 MiniRAG_static-embedding-japanese.ipynb
 POSTGRESQL_USAGE.md
 POSTGRESQL_USAGE_JA.md
 PostgreSQL_on_Colabのサンプルコード.ipynb
'【Ollama-GPT OSS 20B】MiniRAG_on_postgres.ipynb'
 【llamacpp_unsloth_Qwen3.5-27B-GGUF_UD-Q4_K_XL】MiniRAG_on_postgres-Copy1.ipynb


### RAGシステムセットアップ

In [26]:
# テーブルチェックで以下のような表示大量に出るのは正常に動いている証拠
# ---
# PostgreSQL database error: relation "lightrag_doc_full" does not exist
# Failed to check table LIGHTRAG_DOC_FULL in PostgreSQL database
# ---
# SELECT 1 FROM LIGHTRAG_DOC_FULL LIMIT 1
# None
# ---

async def setup_rag_system():
    """RAGシステムを初期化し、準備ができたインスタンスを返す"""

    # 1. データベース設定
    db_config={
        "host": "postgres",    # これは Docker のサービス名「postgres」で指定
        "port": 5432,
        "user": os.getenv("POSTGRES_USER"),
        "password": os.getenv("POSTGRES_PASSWORD"),
        "database": os.getenv("POSTGRES_DB"),
    }

    # 2. PostgreSQLDBインスタンスを作成
    db_postgre = PostgreSQLDB(config=db_config)

    # 3. データベース接続を初期化
    print("------------------------- ポスグレに接続しています -------------------------")
    await db_postgre.initdb()
    print("------------------------- ポスグレに接続しました！ -------------------------")

    # 必要なテーブルが存在するかチェックし、なければ作成する
    print("------------------------- テーブルの存在を確認・作成しています -------------------------")
    await db_postgre.check_tables()
    print("------------------------- テーブルの準備が完了しました！ -------------------------")

    # 5. MiniRAGインスタンスを作成
    os.environ["AGE_GRAPH_NAME"] = "my_minirag_graph" # init_db.sh で指定したもの
    rag = MiniRAG(
        working_dir=WORKING_DIR,
        # llm_model_func=hf_model_complete,
        # llm_model_func=gemini_2_5_flash_complete,
        # llm_model_func=ollama_openai_complete,
        llm_model_func=openai_complete,
        llm_model_max_token_size=1000,
        llm_model_name=LLM_MODEL,
        embedding_func=EmbeddingFunc(
            embedding_dim=EMBEDDING_DIM,
            max_token_size=1000,
            func=lambda texts: hf_embed(
                texts,
                tokenizer=tokenizer,
                embed_model=model
            ),
        ),
        kv_storage="PGKVStorage",
        vector_storage="PGVectorStorage",
        graph_storage="PGGraphStorage",
        doc_status_storage="PGDocStatusStorage",
        vector_db_storage_cls_kwargs={
            "cosine_better_than_threshold": float(os.getenv("COSINE_THRESHOLD"))
        }
    )
    # データベースの情報を渡す
    rag.set_storage_client(db_postgre)    
    return rag

# RAGシステムをセットアップ
try:
    rag = await setup_rag_system()
    print("------------------------- MiniRAGが初期化されました！ -------------------------")
except Exception as e:
    print(f"RAGシステムのセットアップに失敗しました: {e}")

------------------------- ポスグレに接続しています -------------------------
------------------------- ポスグレに接続しました！ -------------------------
------------------------- テーブルの存在を確認・作成しています -------------------------
------------------------- テーブルの準備が完了しました！ -------------------------
------------------------- MiniRAGが初期化されました！ -------------------------


In [27]:
# PostgreSQL database error: relation "lightrag_doc_full" does not exist
# Failed to check table LIGHTRAG_DOC_FULL in PostgreSQL database
# PostgreSQL database error: relation "lightrag_doc_full" does not exist
# PostgreSQL database error: relation "lightrag_doc_chunks" does not exist
# Failed to check table LIGHTRAG_DOC_CHUNKS in PostgreSQL database
# PostgreSQL database error: relation "lightrag_doc_chunks" does not exist
# PostgreSQL database error: relation "lightrag_vdb_entity" does not exist
# Failed to check table LIGHTRAG_VDB_ENTITY in PostgreSQL database
# PostgreSQL database error: relation "lightrag_vdb_entity" does not exist
# PostgreSQL database error: relation "lightrag_vdb_relation" does not exist
# Failed to check table LIGHTRAG_VDB_RELATION in PostgreSQL database
# PostgreSQL database error: relation "lightrag_vdb_relation" does not exist
# PostgreSQL database error: relation "lightrag_llm_cache" does not exist
# Failed to check table LIGHTRAG_LLM_CACHE in PostgreSQL database
# PostgreSQL database error: relation "lightrag_llm_cache" does not exist
# PostgreSQL database error: relation "lightrag_doc_status" does not exist
# Failed to check table LIGHTRAG_DOC_STATUS in PostgreSQL database
# PostgreSQL database error: relation "lightrag_doc_status" does not exist
# ------------------------- ポスグレに接続しています -------------------------
# ------------------------- ポスグレに接続しました！ -------------------------
# ------------------------- テーブルの存在を確認・作成しています -------------------------
# SELECT 1 FROM LIGHTRAG_DOC_FULL LIMIT 1
# None
# SELECT 1 FROM LIGHTRAG_DOC_CHUNKS LIMIT 1
# None
# SELECT 1 FROM LIGHTRAG_VDB_ENTITY LIMIT 1
# None
# SELECT 1 FROM LIGHTRAG_VDB_RELATION LIMIT 1
# None
# SELECT 1 FROM LIGHTRAG_LLM_CACHE LIMIT 1
# None
# SELECT 1 FROM LIGHTRAG_DOC_STATUS LIMIT 1
# None
# ------------------------- テーブルの準備が完了しました！ -------------------------
# ------------------------- MiniRAGが初期化されました！ -------------------------


# 上記の結果になればOK

### 構造化ドキュメントのマルチフィールド挿入と検索

以下の投入しているデータが悪すぎて検索のテストにならないが、雰囲気的にはうまく実装できていそうなので、一旦OKとする


In [28]:
from datetime import datetime, timezone

workspace = rag.storage_client.workspace

structured_docs = [
    {
        "workspace": "procurement_docs",
        "doc_id": "proc-plan-fy2026-apac",
        "title": "2026年度 APAC地域包括調達戦略・実施計画書（第1版）",
        "summary": "2026年度から2028年度までの中期経営計画に基づき、APAC地域における調達オペレーションの最適化、サプライヤー・レジリエンスの強化、およびコスト構造の改革を定義する包括的文書。",
        "body": ["""# 1. はじめに
本計画書は、地政学的リスクの増大と原材料価格の変動に対応するため、2026年度におけるAPAC地域の調達方針を定めるものである。

# 2. 戦略目標
2026年度の最優先課題は「供給網の安定化」と「持続可能なコスト削減」の両立である。具体的には以下のKPIを設定する。
- 重点カテゴリーにおける複数社購買（Multi-sourcing）率を現状の40%から65%へ引き上げる。
- 地域内調達率（LCR: Local Content Ratio）を平均12%向上させ、物流コストと関税リスクを低減する。
- AIによる需給予測モデルを本格稼働させ、過剰在庫による保管コストを年間で約1.2億円削減する。

# 3. サプライヤーマネジメント
## 3.1 新規サプライヤーの開拓
東南アジア諸国（特にベトナム、タイ、インドネシア）における製造拠点の拡大に伴い、現地の有力サプライヤー50社との戦略的パートナーシップを締結する。
特にインド市場においては、半導体関連部材の現地調達比率を25%まで高めることを目指す。

## 3.2 ESG対応と監査
2026年度より、全ての主要サプライヤー（年間取引額1億円以上）に対し、CO2排出量の四半期報告を義務付ける。
また、人権デューデリジェンスに関する第三者機関による監査を、対象企業の30%で実施する。

# 4. デジタル変革（Procurement DX）
調達プロセスの透明性を高めるため、次世代型E-Procurementシステムを全拠点に導入する。
これにより、発注から支払いまでのリードタイムを20%短縮し、サプライヤー支払いの100%電子化を実現する。

# 5. リスク管理
地政学的リスクに対応するため、特定の1カ国に生産が集中している重要部材（23品目）について、代替生産拠点の確保をQ2（7-9月）までに完了させる。
大規模災害発生時の緊急調達フローを再構築し、初動対応時間を現状の24時間から6時間以内へ短縮する。"""
        ],
        "status": "in_progress",
        "region": "APAC",
        "priority": 1,
        "created_at": datetime(2025, 10, 1, 8, 30, tzinfo=timezone.utc),
        "metadata": {
            "category": "planning_document",
            "region": "APAC",
            "year": 2026,
            "owner_department": "IT Procurement Department",
            "approver": "John Smith",
            "version": "2.1"
        },
    },
    {
        "workspace": "procurement_docs",
        "doc_id": "contract-na-fy2025-qsi",
        "title": "2026年度 次世代ITサービス調達・運用ガイドライン",
        "summary": "グローバル全拠点におけるIT資産、ソフトウェア、クラウドサービスの調達基準とセキュリティ要件。",
        "body": ["""# 1. 目的
本ガイドラインは、グループ全体のITガバナンスを強化し、サイバーセキュリティリスクを低減しつつ、ITコストの最適化を図ることを目的とする。

# 2. クラウドファースト戦略
新規ITシステムの導入に際しては、クラウドネイティブな構成を原則とする。
- 推奨プラットフォーム：AWS (Asia Pacific Regions), Microsoft Azure (Global)
- 調達基準：可用性99.99%以上の保証、およびISO 27017認証の取得が必須。

# 3. ソフトウェア・ライセンス管理
SaaS利用の急増に伴い、シャドーITの撲滅とライセンス費用の重複を排除する。
- 全拠点共通のSaaS管理ツールを導入し、利用率が30%以下のアプリケーションについては契約更新を行わない。
- オープンソースソフトウェア（OSS）の利用に関しては、法務部およびセキュリティ部門の承認が必要。

# 4. セキュリティ要件（サプライチェーンセキュリティ）
ITサービス提供ベンダーに対しては、以下のセキュリティ基準への準拠を求める。
1. SOC2 Type2レポートの定期的な提出。
2. ゼロトラストアーキテクチャに基づいたアクセス制御の実装。
3. 脆弱性診断を年2回以上実施し、その結果を報告すること。

# 5. ハードウェア調達と循環経済
PC、サーバー、ネットワーク機器の調達において、環境負荷を最小限に抑える。
- 廃棄されるハードウェアの80%以上をリサイクルまたはリユースする「IT資産循環プログラム」をQ3より開始する。
- 消費電力効率が前世代比で15%以上向上しているモデルを優先的に選定する。"""
        ],
        "status": "finalized",
        "region": "NA",
        "priority": 1,
        "created_at": datetime(2025, 10, 2, 9, 15, tzinfo=timezone.utc),
        "metadata": {
            "category": "contract",
            "region": "NA",
            "year": 2025,
            "owner_department": "Legal & Supply Chain Management",
            "supplier_name": "Quantum Systems Inc.",
            "contract_value_usd": 12000000
        },
    },
]

schema = {
    "table": "public.customer_orders",
    "id_column": "doc_id",
    "fields": {
        "workspace": {"type": "text", "nullable": False},
        "doc_id": {"type": "text", "nullable": False},
        "title": {"type": "text"},
        "summary": {"type": "text"},
        "body": {"type": "text"},
        "status": {"type": "text"},
        "region": {"type": "text"},
        "priority": {"type": "integer"},
        "created_at": {"type": "timestamp"},
    },
    "conflict_columns": ["workspace", "doc_id"],
}

await rag.ainsert(structured_docs, schema=schema, text_fields=["title", "summary", "body"])
print(f"構造化ドキュメントを {len(structured_docs)} 件登録しました (workspace={workspace}).")

🚀 AINSERT called with overwrite=False
📥 Input: 2 documents
📥 IDs: ['proc-plan-fy2026-apac', 'contract-na-fy2025-qsi']
📥 Metadatas: [{'category': 'planning_document', 'region': 'APAC', 'year': 2026, 'owner_department': 'IT Procurement Department', 'approver': 'John Smith', 'version': '2.1', 'workspace': 'procurement_docs', 'title': '2026年度 APAC地域包括調達戦略・実施計画書（第1版）', 'summary': '2026年度から2028年度までの中期経営計画に基づき、APAC地域における調達オペレーションの最適化、サプライヤー・レジリエンスの強化、およびコスト構造の改革を定義する包括的文書。', 'body': '# 1. はじめに\n本計画書は、地政学的リスクの増大と原材料価格の変動に対応するため、2026年度におけるAPAC地域の調達方針を定めるものである。\n\n# 2. 戦略目標\n2026年度の最優先課題は「供給網の安定化」と「持続可能なコスト削減」の両立である。具体的には以下のKPIを設定する。\n- 重点カテゴリーにおける複数社購買（Multi-sourcing）率を現状の40%から65%へ引き上げる。\n- 地域内調達率（LCR: Local Content Ratio）を平均12%向上させ、物流コストと関税リスクを低減する。\n- AIによる需給予測モデルを本格稼働させ、過剰在庫による保管コストを年間で約1.2億円削減する。\n\n# 3. サプライヤーマネジメント\n## 3.1 新規サプライヤーの開拓\n東南アジア諸国（特にベトナム、タイ、インドネシア）における製造拠点の拡大に伴い、現地の有力サプライヤー50社との戦略的パートナーシップを締結する。\n特にインド市場においては、半導体関連部材の現地調達比率を25%まで高めることを目指す。\n\n## 3.2 ESG対応と監査\n2026年度より、全ての主要サプ

Generating embeddings: 100%|██████████| 1/1 [00:00<00:00,  4.58batch/s]


💾 Upserting chunk 'chunk-body-00424...' with metadata: {'year': 2025, 'region': 'NA', 'category': 'contract', 'supplier_name': 'Quantum Systems Inc.', 'owner_department': 'Legal & Supply Chain Management', 'contract_value_usd': 12000000, 'priority': 1, 'text_field': 'body'}
💾 Upserting chunk 'chunk-title-e215...' with metadata: {'year': 2025, 'region': 'NA', 'category': 'contract', 'supplier_name': 'Quantum Systems Inc.', 'owner_department': 'Legal & Supply Chain Management', 'contract_value_usd': 12000000, 'priority': 1, 'text_field': 'title'}
💾 Upserting chunk 'chunk-region-b28...' with metadata: {'year': 2025, 'region': 'NA', 'category': 'contract', 'supplier_name': 'Quantum Systems Inc.', 'owner_department': 'Legal & Supply Chain Management', 'contract_value_usd': 12000000, 'priority': 1, 'text_field': 'region'}
💾 Upserting chunk 'chunk-status-03b...' with metadata: {'year': 2025, 'region': 'NA', 'category': 'contract', 'supplier_name': 'Quantum Systems Inc.', 'owner_department': '

Generating embeddings: 100%|██████████| 2/2 [00:00<00:00, 12.91batch/s]


⚙️  Processing doc 'proc-plan-fy2026-apac', status_doc.metadata = {"body": "# 1. はじめに\n本計画書は、地政学的リスクの増大と原材料価格の変動に対応するため、2026年度におけるAPAC地域の調達方針を定めるものである。\n\n# 2. 戦略目標\n2026年度の最優先課題は「供給網の安定化」と「持続可能なコスト削減」の両立である。具体的には以下のKPIを設定する。\n- 重点カテゴリーにおける複数社購買（Multi-sourcing）率を現状の40%から65%へ引き上げる。\n- 地域内調達率（LCR: Local Content Ratio）を平均12%向上させ、物流コストと関税リスクを低減する。\n- AIによる需給予測モデルを本格稼働させ、過剰在庫による保管コストを年間で約1.2億円削減する。\n\n# 3. サプライヤーマネジメント\n## 3.1 新規サプライヤーの開拓\n東南アジア諸国（特にベトナム、タイ、インドネシア）における製造拠点の拡大に伴い、現地の有力サプライヤー50社との戦略的パートナーシップを締結する。\n特にインド市場においては、半導体関連部材の現地調達比率を25%まで高めることを目指す。\n\n## 3.2 ESG対応と監査\n2026年度より、全ての主要サプライヤー（年間取引額1億円以上）に対し、CO2排出量の四半期報告を義務付ける。\nまた、人権デューデリジェンスに関する第三者機関による監査を、対象企業の30%で実施する。\n\n# 4. デジタル変革（Procurement DX）\n調達プロセスの透明性を高めるため、次世代型E-Procurementシステムを全拠点に導入する。\nこれにより、発注から支払いまでのリードタイムを20%短縮し、サプライヤー支払いの100%電子化を実現する。\n\n# 5. リスク管理\n地政学的リスクに対応するため、特定の1カ国に生産が集中している重要部材（23品目）について、代替生産拠点の確保をQ2（7-9月）までに完了させる。\n大規模災害発生時の緊急調達フローを再構築し、初動対応時間を現状の24時間から6時間以内へ短縮する。", "year": 2026, "title": "2026年度 APAC地域包括調達戦

...

💾 Upserting chunk 'chunk-body-af651...' with metadata: {'year': 2026, 'region': 'APAC', 'version': '2.1', 'approver': 'John Smith', 'category': 'planning_document', 'owner_department': 'IT Procurement Department', 'priority': 1, 'text_field': 'body'}


Generating embeddings: 100%|██████████| 1/1 [00:00<00:00,  4.32batch/s]


💾 Upserting chunk 'chunk-title-171b...' with metadata: {'year': 2026, 'region': 'APAC', 'version': '2.1', 'approver': 'John Smith', 'category': 'planning_document', 'owner_department': 'IT Procurement Department', 'priority': 1, 'text_field': 'title'}
💾 Upserting chunk 'chunk-region-724...' with metadata: {'year': 2026, 'region': 'APAC', 'version': '2.1', 'approver': 'John Smith', 'category': 'planning_document', 'owner_department': 'IT Procurement Department', 'priority': 1, 'text_field': 'region'}
💾 Upserting chunk 'chunk-status-138...' with metadata: {'year': 2026, 'region': 'APAC', 'version': '2.1', 'approver': 'John Smith', 'category': 'planning_document', 'owner_department': 'IT Procurement Department', 'priority': 1, 'text_field': 'status'}
💾 Upserting chunk 'chunk-summary-6e...' with metadata: {'year': 2026, 'region': 'APAC', 'version': '2.1', 'approver': 'John Smith', 'category': 'planning_document', 'owner_department': 'IT Procurement Department', 'priority': 1, 'text_field':

ConnectionError: llama-server に接続できません。base_url=http://172.17.0.1:1067/v1 / 試行URL=http://172.17.0.1:1067/v1/health / 原因=ConnectTimeout('')

⠴ Processed 5 chunks, 25 entities(duplicated), 29 relations(duplicated)

In [29]:
import json

doc_ids = [doc["doc_id"] for doc in structured_docs]
chunk_rows = await rag.storage_client.query(
    """
    SELECT full_doc_id,
           metadata,
           metadata->>'text_field' AS text_field,
           content,
           chunk_order_index
    FROM LIGHTRAG_DOC_CHUNKS
    WHERE workspace=$1
      AND full_doc_id = ANY($2::text[])
    ORDER BY full_doc_id, chunk_order_index
    """,
    params={"workspace": workspace, "doc_ids": doc_ids},
    multirows=True,
)

print(f"取得チャンク数: {len(chunk_rows)}")
for row in chunk_rows:
    head = row["content"].split("\n")[0][:60]
    metadata_json = json.dumps(row["metadata"], ensure_ascii=False)
    print(f"- {row['full_doc_id']} | text_field={row['text_field']} | chunk_order={row['chunk_order_index']}\n  preview: {head}\n  metadata: {metadata_json}")


取得チャンク数: 16
- contract-na-fy2025-qsi | text_field=workspace | chunk_order=0
  preview: procurement_docs
  metadata: "{\"year\": 2025, \"region\": \"NA\", \"category\": \"contract\", \"priority\": 1, \"text_field\": \"workspace\", \"supplier_name\": \"Quantum Systems Inc.\", \"owner_department\": \"Legal & Supply Chain Management\", \"contract_value_usd\": 12000000}"
- contract-na-fy2025-qsi | text_field=_all | chunk_order=0
  preview: # 1. 目的
  metadata: "{\"year\": 2025, \"region\": \"NA\", \"category\": \"contract\", \"priority\": 1, \"text_field\": \"_all\", \"supplier_name\": \"Quantum Systems Inc.\", \"owner_department\": \"Legal & Supply Chain Management\", \"contract_value_usd\": 12000000}"
- contract-na-fy2025-qsi | text_field=body | chunk_order=0
  preview: # 1. 目的
  metadata: "{\"year\": 2025, \"region\": \"NA\", \"category\": \"contract\", \"priority\": 1, \"text_field\": \"body\", \"supplier_name\": \"Quantum Systems Inc.\", \"owner_department\": \"Legal & Supply Chain Mana

#### target_fields を利用した検索例


In [30]:
from pprint import pprint

summary_param = QueryParam(
    mode="light",
    target_fields=["summary"],
    include_provenance=True,
    top_k=3,
)

queries = [
    "2026年度のインド市場における調達目標は何ですか？",
    "ITサービス調達におけるクラウドプラットフォームの選定基準を教えてください。",
    "サプライヤーに対するESGやCO2排出に関する要求事項は何ですか？"
]

for query in queries:
    print(f"\n質問: {query}")
    summary_result, summary_sources = await rag.aquery(
        query,
        param=summary_param,
    )
    print("回答:")
    if isinstance(summary_result, dict):
        pprint(summary_result)
    else:
        print(summary_result)
    
    print("\n参照したチャンク:")
    for src in summary_sources:
        print("-", src.split("\n")[0])


質問: 2026年度のインド市場における調達目標は何ですか？
🎯 Using distance threshold: -1.0
📊 Query returned 3 results
✅ Result 1: id=ent-8cbded0be35c..., distance=0.30618952962954304
✅ Result 2: id=ent-0a4f93a5b727..., distance=0.2765268762324221
🔎 Total records in LIGHTRAG_VDB_ENTITY: 58
🔎 Raw metadata and distance values:
   - ID: ent-8cbded0be35c...
     Raw metadata: {'year': 2025, 'region': 'NA', 'category': 'contract', 'priority': 1, 'text_field': 'title', 'supplier_name': 'Quantum Systems Inc.', 'owner_department': 'Legal & Supply Chain Management', 'contract_value_usd': 12000000}
     Extracted category: contract
     Distance: 0.30618952962954304
   - ID: ent-0a4f93a5b727...
     Raw metadata: {'year': 2025, 'region': 'NA', 'category': 'contract', 'priority': 1, 'text_field': 'title', 'supplier_name': 'Quantum Systems Inc.', 'owner_department': 'Legal & Supply Chain Management', 'contract_value_usd': 12000000}
     Extracted category: contract
     Distance: 0.2765268762324221
   - ID: Ename-9da118c441

#### target_fields と metadata_filter の併用


In [34]:
supply_param = QueryParam(
    mode="light",
    target_fields=["body"],
    metadata_filter={"category": "supply", "year": 2025},
    include_provenance=True,
    only_need_context=True,
    top_k=3,
)
queries = [
    "2026年度のインド市場における調達目標は何ですか？",
    "ITサービス調達におけるクラウドプラットフォームの選定基準を教えてください。",
    "サプライヤーに対するESGやCO2排出に関する要求事項は何ですか？"
]

for query in queries:
    print(f"\n質問: {query}")
    supply_context, supply_sources = await rag.aquery(
        query,
        param=supply_param,
    )
    
    print("取得したコンテキスト:")
    if isinstance(supply_context, dict):
        pprint(supply_context)
    else:
        print(supply_context)
    
    print("\n対象チャンク (body フィールド):")
    for src in supply_sources:
        print("-", src.split("\n")[0])


質問: 2026年度のインド市場における調達目標は何ですか？
🎯 Using distance threshold: -1.0
📊 Query returned 3 results
✅ Result 1: id=Ename-4ecfeeae37..., distance=0.6822239236807861
✅ Result 2: id=Ename-773fbe264b..., distance=0.6318540111274639
🔎 Total records in LIGHTRAG_VDB_ENTITY: 90
🔎 Raw metadata and distance values:
   - ID: Ename-4ecfeeae37...
     Raw metadata: {'year': 2025, 'region': 'NA', 'category': 'contract', 'priority': 1, 'text_field': '_all', 'supplier_name': 'Quantum Systems Inc.', 'owner_department': 'Legal & Supply Chain Management', 'contract_value_usd': 12000000}
     Extracted category: contract
     Distance: 0.6822239236807861
   - ID: Ename-773fbe264b...
     Raw metadata: {'year': 2026, 'region': 'APAC', 'version': '2.1', 'approver': 'John Smith', 'category': 'planning_document', 'priority': 1, 'text_field': 'summary', 'owner_department': 'IT Procurement Department'}
     Extracted category: planning_document
     Distance: 0.6318540111274639
   - ID: ent-4ecfeeae37cc...
     Raw met

#### 時間フィルタ（`start_time` / `end_time`）の利用

In [20]:
from datetime import datetime, timedelta, timezone

now = datetime.now(timezone.utc)

param = QueryParam(
    mode="light",
    start_time=now.isoformat(),
    end_time=(now + timedelta(hours=1)).isoformat(),
    metadata_filter={"category": "plan"},
)

answer, sources = await rag.aquery("最近登録された計画について教えて", param=param)
print(answer)
print(sources)

🎯 Using distance threshold: -1.0
📊 Query returned 0 results
🔎 Total records in LIGHTRAG_VDB_ENTITY: 74
🔎 Raw metadata and distance values:
   - ID: Ename-a84fbb5cff...
     Raw metadata: {'year': 2026, 'region': 'APAC', 'version': '2.1', 'approver': 'John Smith', 'category': 'planning_document', 'priority': 1, 'text_field': '_all', 'owner_department': 'IT Procurement Department'}
     Extracted category: planning_document
     Distance: 0.36557339889926355
   - ID: Ename-4161566299...
     Raw metadata: {'year': 2026, 'region': 'APAC', 'version': '2.1', 'approver': 'John Smith', 'category': 'planning_document', 'priority': 1, 'text_field': '_all', 'owner_department': 'IT Procurement Department'}
     Extracted category: planning_document
     Distance: 0.3612903468224189
   - ID: Ename-44258c97dc...
     Raw metadata: {'year': 2025, 'region': 'NA', 'category': 'contract', 'priority': 1, 'text_field': 'created_at', 'supplier_name': 'Quantum Systems Inc.', 'owner_department': 'Legal & Su

#### target_fields を指定しない場合（_all チャンク）

In [35]:
default_param = QueryParam(mode="light", top_k=3)
default_answer, default_sources = await rag.aquery(
    "契約全体の概要をまとめてください。",
    param=default_param,
)

print("回答:")
print(default_answer)

print("\n参照したソース: ")
for src in default_sources:
    print(src)
    print("================================================")


🎯 Using distance threshold: -1.0
📊 Query returned 3 results
✅ Result 1: id=Ename-773fbe264b..., distance=0.30495817211176424
✅ Result 2: id=ent-a35eb4dd7a5b..., distance=0.28456572084164833
🔎 Total records in LIGHTRAG_VDB_ENTITY: 90
🔎 Raw metadata and distance values:
   - ID: Ename-773fbe264b...
     Raw metadata: {'year': 2026, 'region': 'APAC', 'version': '2.1', 'approver': 'John Smith', 'category': 'planning_document', 'priority': 1, 'text_field': 'summary', 'owner_department': 'IT Procurement Department'}
     Extracted category: planning_document
     Distance: 0.30495817211176424
   - ID: ent-a35eb4dd7a5b...
     Raw metadata: {'year': 2026, 'region': 'APAC', 'version': '2.1', 'approver': 'John Smith', 'category': 'planning_document', 'priority': 1, 'text_field': 'body', 'owner_department': 'IT Procurement Department'}
     Extracted category: planning_document
     Distance: 0.28456572084164833
   - ID: Ename-4b6d4ed759...
     Raw metadata: {'year': 2026, 'region': 'APAC', 've

### フィルターしないシンプルVerで試してみる

In [31]:
# 良い感じにできている
default_param = QueryParam(
    mode="light",
    include_provenance=True,
    top_k=3,
)

queries = [
    "2026年度のインド市場における調達目標は何ですか？",
    "ITサービス調達におけるクラウドプラットフォームの選定基準を教えてください。",
    "サプライヤーに対するESGやCO2排出に関する要求事項は何ですか？"
]

for query in queries:
    print(f"\n質問: {query}")
    default_result, default_sources = await rag.aquery(
        query,
        param=default_param,
    )
    print("回答:")
    if isinstance(default_result, dict):
        pprint(default_result)
    else:
        print(default_result)
    
    print("\n参照したチャンク:")
    for src in default_sources:
        print("-", src.split("\n")[0])


質問: 2026年度のインド市場における調達目標は何ですか？
🎯 Using distance threshold: -1.0
📊 Query returned 3 results
✅ Result 1: id=ent-8cbded0be35c..., distance=0.3001357979049961
✅ Result 2: id=ent-0a4f93a5b727..., distance=0.28975188099571736
🔎 Total records in LIGHTRAG_VDB_ENTITY: 58
🔎 Raw metadata and distance values:
   - ID: ent-8cbded0be35c...
     Raw metadata: {'year': 2025, 'region': 'NA', 'category': 'contract', 'priority': 1, 'text_field': 'title', 'supplier_name': 'Quantum Systems Inc.', 'owner_department': 'Legal & Supply Chain Management', 'contract_value_usd': 12000000}
     Extracted category: contract
     Distance: 0.3001357979049961
   - ID: ent-0a4f93a5b727...
     Raw metadata: {'year': 2025, 'region': 'NA', 'category': 'contract', 'priority': 1, 'text_field': 'title', 'supplier_name': 'Quantum Systems Inc.', 'owner_department': 'Legal & Supply Chain Management', 'contract_value_usd': 12000000}
     Extracted category: contract
     Distance: 0.28975188099571736
   - ID: Ename-9da118c441

In [37]:
# ミニモードも試してみる → 精度は置いておいてちゃんと動いていそう
default_param = QueryParam(
    mode="mini",
    include_provenance=True,
    top_k=3,
)

queries = [
    "2026年度のインド市場における調達目標は何ですか？",
    "ITサービス調達におけるクラウドプラットフォームの選定基準を教えてください。",
    "サプライヤーに対するESGやCO2排出に関する要求事項は何ですか？"
]

for query in queries:
    print(f"\n質問: {query}")
    default_result, default_sources = await rag.aquery(
        query,
        param=default_param,
    )
    print("回答:")
    if isinstance(default_result, dict):
        pprint(default_result)
    else:
        print(default_result)
    
    print("\n参照したチャンク:")
    for src in default_sources:
        print("-", src.split("\n")[0])


質問: 2026年度のインド市場における調達目標は何ですか？
★デバッグ(minirag.py): mini mode
★デバッグ(postgres_impl.py): create_graph already exists
🎯 Using distance threshold: -1.0
📊 Query returned 3 results
✅ Result 1: id=Ename-773fbe264b..., distance=0.6766635066308787
✅ Result 2: id=Ename-4ecfeeae37..., distance=0.6096748005065683
🔎 Total records in LIGHTRAG_DOC_CHUNKS: 16
🔎 Raw metadata and distance values:
   - ID: chunk-summary-6e...
     Raw metadata: {'year': 2026, 'region': 'APAC', 'version': '2.1', 'approver': 'John Smith', 'category': 'planning_document', 'priority': 1, 'text_field': 'summary', 'owner_department': 'IT Procurement Department'}
     Extracted category: planning_document
     Distance: 0.08200306450534867
   - ID: chunk-title-171b...
     Raw metadata: {'year': 2026, 'region': 'APAC', 'version': '2.1', 'approver': 'John Smith', 'category': 'planning_document', 'priority': 1, 'text_field': 'title', 'owner_department': 'IT Procurement Department'}
     Extracted category: planning_document
     D

In [32]:
# 約12分かかった
import time
start_time = time.time()

# サンプルテキストデータ
sample_texts = [
    """
今日は素晴らしい一日でした。朝早く起きて、近所の公園を散歩しました。
桜の花が満開で、とても美しかったです。午後は友人と映画を見に行きました。
「君の名は。」という映画で、とても感動的でした。
夜は家族と一緒に夕食を取り、楽しい時間を過ごしました。
""",
    """
昨日は仕事で大きなプロジェクトが完了しました。
チーム全員で3ヶ月間取り組んできたAIシステムの開発が終わりました。
機械学習モデルの精度が95%を超え、クライアントからも高い評価をいただきました。
今夜はチームメンバーと祝賀会を開く予定です。
""",
    """
週末は料理に挑戦しました。初めてパスタを一から作ってみました。
小麦粉から麺を作るのは思っていたより難しかったですが、
最終的にはとても美味しいカルボナーラができました。
次回はリゾットに挑戦してみたいと思います。
""",
    """
読書が趣味で、最近は村上春樹の「ノルウェイの森」を読んでいます。
主人公の心情描写がとても繊細で、引き込まれます。
また、技術書も読んでおり、「深層学習」について学んでいます。
理論と実践のバランスが取れた良い本だと思います。
"""
]

# データの挿入
print("データを挿入中...")

async def insert_texts(rag_instance, texts):
    for i, text in enumerate(texts):
        print(f"テキスト {i+1}/{len(texts)} を挿入中...")
        await rag_instance.ainsert(text.strip())   # overwrite は デフォルトは False 

    print("\nすべてのデータが挿入されました！")


# イベントループが既に実行中の場合
try:
    await insert_texts(rag, sample_texts)
except RuntimeError:
    # 新しいループで実行
    asyncio.run(insert_texts(rag, sample_texts))

end_time = time.time()
elapsed_time = end_time - start_time
print(f"処理時間: {elapsed_time:.4f}秒")

データを挿入中...
テキスト 1/4 を挿入中...
🚀 AINSERT called with overwrite=False
📥 Input: 1 documents
📥 IDs: None
📥 Metadatas: None
📝 Storing doc 'doc-6ecfce929dc1037e02183d8a074dd3d9' with metadata: {}
⚙️  Processing doc 'doc-6ecfce929dc1037e02183d8a074dd3d9', status_doc.metadata = {}
📦 Created 1 standard chunks for doc 'doc-6ecfce929dc1037e02183d8a074dd3d9'
   └─ Chunk 'chunk-3c79463c8d...' text_field: _all, metadata: {'text_field': '_all'}


Generating embeddings: 100%|██████████| 1/1 [00:00<00:00, 43.04batch/s]

💾 Upserting chunk 'chunk-3c79463c8d...' with metadata: {'text_field': '_all'}


⠙ Processed 1 chunks, 8 entities(duplicated), 8 relations(duplicated)
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): crea

Generating embeddings: 100%|██████████| 1/1 [00:00<00:00, 30.18batch/s]


...

Generating embeddings: 100%|██████████| 1/1 [00:00<00:00, 90.32batch/s]

💾 Upserting chunk 'chunk-4091c9fc91...' with metadata: {'text_field': '_all'}


⠙ Processed 1 chunks, 4 entities(duplicated), 5 relations(duplicated)
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): crea

Generating embeddings: 100%|██████████| 1/1 [00:00<00:00, 46.60batch/s]

KG successfully indexed.

すべてのデータが挿入されました！
処理時間: 341.7114秒


In [33]:
# 約3分かかった

def print_test_case(name: str, answer: str, source: str, ):
    """テストケースの結果を分かりやすく表示するヘルパー関数"""
    print(f"\n--- {name} ---")
    print("Answer:", answer)
    print("Source:", source)
    print("-" * (len(name) + 8))


# # サンプルクエリ
# queries = [
#     "映画について教えて",
#     "仕事のプロジェクトはどうでしたか？",
#     "料理で何を作りましたか？",
#     "読んでいる本について教えて",
#     "散歩について詳しく教えて"
# ]

# # 各モードでクエリを実行。この3つがある
# modes = ["naive", "mini", "light"]

# for query in queries:
#     print(f"\n{'='*50}")
#     print(f"クエリ: {query}")
#     print(f"{'='*50}")

#     for mode in modes:
#         print(f"\n--- {mode.upper()}モード ---")
#         try:
#             answer = rag.query(query, param=QueryParam(mode=mode))     # .replace("\n", "").replace("\r", "")
#             print(f"回答: {answer}")
#         except Exception as e:
#             print(f"エラー: {e}")


async def run_queries(rag_instance, queries):
    # 各モードでクエリを実行。この3つがある
    modes = ["naive", "mini", "light"]

    for query in queries:
        print(f"\n{'='*50}")
        print(f"クエリ: {query}")
        print(f"{'='*50}")

        for mode in modes:
            print(f"\n--- {mode.upper()}モード ---")
            try:
                # 非同期でクエリを実行
                _answer, _source = await rag_instance.aquery(query, param=QueryParam(mode=mode))
                print_test_case(mode+"モード", _answer, _source)
            except Exception as e:
                print(f"エラー: {e}")


start_time = time.time()

sample_queries = [
    "映画について教えて",
    "仕事のプロジェクトはどうでしたか？",
    "料理で何を作りましたか？",
    "読んでいる本について教えて",
    "散歩について詳しく教えて"
]

# イベントループが既に実行中の場合
try:
    await run_queries(rag, sample_queries)
except RuntimeError:
    # 新しいループで実行
    asyncio.run(run_queries(rag, sample_queries))


end_time = time.time()
elapsed_time = end_time - start_time
print(f"処理時間: {elapsed_time:.4f}秒")


クエリ: 映画について教えて

--- NAIVEモード ---
🎯 Using distance threshold: -1.0
🔍 No metadata filter applied
📊 Query returned 20 results
✅ Result 1: id=chunk-cf733b0e4f..., distance=0.4565610454279263
✅ Result 2: id=chunk-23680af8dd..., distance=0.45122152041276387
🔎 Total records in LIGHTRAG_DOC_CHUNKS: 20
🔎 Records with metadata: 20
🔎 Raw metadata and distance values:
   - ID: chunk-cf733b0e4f...
     Raw metadata: {'text_field': '_all'}
     Extracted category: None
     Distance: 0.4565610454279263
   - ID: chunk-23680af8dd...
     Raw metadata: {'text_field': '_all'}
     Extracted category: None
     Distance: 0.45122152041276387
   - ID: chunk-4091c9fc91...
     Raw metadata: {'text_field': '_all'}
     Extracted category: None
     Distance: 0.3793197605176075

--- naiveモード ---
Answer: 提供された文書に基づくと、映画に関する具体的な情報は1つだけ記載されています。

ある日、午後に友人と映画館を訪れ、映画「君の名は。」を鑑賞したという記述があります。この作品は非常に感動的であったと評価されており、その日は朝から桜の満開の公園を散歩するなど、素晴らしい一日だったと振り返られています。

ただし、提供された資料の大部分は、2026年度におけるAPAC地域の調達戦略計画書や、次世代ITサービスの調達・

## ここからは metadata_filter と time_filter を使ったテスト

## 初回も2回目もうまくいった(2回目というのは以下のセルを2回実行するということ)
- 精度にブレがあり、回答できる時とできない時があるが、今回は精度は求めていないのでこれで OK とする

In [42]:
from datetime import datetime, timedelta
from typing import Optional, Dict, List, Any

async def run_tests():
    """RAGシステムのフィルタリング機能をテストするメイン関数"""
    # 0. RAGシステムのセットアップ
    try:
        rag_with_filter = await setup_rag_system()
        print("------------------------- RAGシステムが初期化されました！ -------------------------")
    except Exception as e:
        print(f"RAGシステムのセットアップに失敗しました: {e}")
        return

    # 1. テストデータの準備と登録
    # タイムスタンプのテストのため、登録を複数回に分ける
    print("\n[ステップ1] データの登録を開始します...")

    # データセット1
    docs1 = [
        {"doc_id": "doc1", "content": "今日は東京でとても良い天気です。", "metadata": {"category": "weather", "city": "Tokyo", "year": 2024}},
        {"doc_id": "doc2", "content": "昨日の大阪は雨でした。", "metadata": {"category": "weather", "city": "Osaka", "year": 2024}},
    ]
    await rag_with_filter.ainsert(
        input=[d["content"] for d in docs1],
        ids=[d["doc_id"] for d in docs1],
        metadatas=[d["metadata"] for d in docs1],
        overwrite=True
    )
    print("データセット1 (doc1, doc2) を登録しました。")
    
    time_after_docs1 = datetime.utcnow()  # → Postgres は UTC で解釈するため、必ず UTC にすること
    await asyncio.sleep(10)  # タイムスタンプを明確に区別するため10秒待機
    
    # データセット2
    docs2 = [
        {"doc_id": "doc3", "content": "日本の首都は東京です。最近は兵庫も主要な都市に入るか議論されています。", "metadata": {"category": "geography", "country": "Japan", "year": 2023}},
        {"doc_id": "doc4", "content": "大阪は日本の主要な都市の一つです。最近は秋田も主要な都市に入るか議論されています。", "metadata": {"category": "geography", "country": "Japan", "year": 2023}},
        {"doc_id": "doc5", "content": "What is the capital of Japan? It's Tokyo.", "metadata": {}},  # メタデータなし
    ]
    await rag_with_filter.ainsert(
        input=[d["content"] for d in docs2],
        ids=[d["doc_id"] for d in docs2],
        metadatas=[d["metadata"] for d in docs2],
        overwrite=True
    )
    print("データセット2 (doc3, doc4, doc5) を登録しました。")
    time_after_docs2 = datetime.utcnow()  # → Postgres は UTC で解釈するため、必ず UTC にすること

    print("\n[ステップ2] 検索フィルタリングのテストを実行します...")

    # 思考プロセスと回答言語の指示
    instruction = "英語で思考して日本語で回答してください。回答は情報源に基づいて回答し、回答できない場合は`回答できない`と答えてください。"

    # --- メタデータフィルタリングのテスト ---
    param1 = QueryParam(mode="light",
                        metadata_filter={"category": "weather"})
    query1 = f"今日の天気について教えてください。{instruction}"
    answer1, source1 = await rag_with_filter.aquery(query1, param=param1)
    print_test_case("Test Case 1: 単一メタデータでフィルタ ('weather')", answer1, source1)

    param2 = QueryParam(mode="light",
                        metadata_filter={"category": "geography", "country": "Japan"})
    query2 = f"日本の地理に関する情報を情報源に基づいて回答してください。{instruction}"
    answer2, source2 = await rag_with_filter.aquery(query2, param=param2)
    print_test_case("Test Case 2: 複数メタデータでフィルタ (AND条件)。ちゃんと成功したのでフィルター機能は正常かも。ただ、ちょっと精度が低い", answer2, source2,)

    param3 = QueryParam(mode="light",
                        metadata_filter={"city": "北海道"})
    query3 = f"北海道の天気はどうですか？{instruction}"
    answer3, source3 = await rag_with_filter.aquery(query3, param=param3)
    print_test_case("Test Case 3: 存在しないメタデータ値でフィルタ（失敗例）", answer3, source3)

    query4 = f"東京について何か知っていますか？{instruction}"
    answer4, source4 = await rag_with_filter.aquery(query4) # フィルタなし
    print_test_case("Test Case 4: フィルタなしで検索", answer4, source4)

    param5 = QueryParam(mode="light",
                        metadata_filter={"category": "weather"})
    query5 = f"今日 の 東京 の 天気 について詳しく教えてください。{instruction}" # 'Tokyo'はdoc5にも含まれるが、こっちはフィルタで除外されるはず
    answer5, source5 = await rag_with_filter.aquery(query5, param=param5)
    print_test_case("Test Case 5: メタデータフィルタはメタデータなし文書を除外(正解: 天気について回答できる。これ精度が低い気がする)", answer5, source5)

    param6 = QueryParam(mode="light",
                        metadata_filter={"year": 2023})
    query6 = f"2023年の日本の出来事について何か知っていますか？{instruction}"
    answer6, source6 = await rag_with_filter.aquery(query6, param=param6)
    print_test_case("Test Case 6: 数値のメタデータでフィルタ", answer6, source6)

    # --- 時間フィルタリングのテスト ---
    param7 = QueryParam(mode="light",
                        start_time=time_after_docs1.isoformat())
    query7 = f"日本に関する最近登録された情報を教えてください。{instruction}"
    answer7, source7 = await rag_with_filter.aquery(query7, param=param7)
    print_test_case("Test Case 7: 'start_time'でフィルタ (後半の登録データのみ)。正解: 後半のデータのみで回答できること", answer7, source7)

    param8 = QueryParam(mode="light",
                        end_time=time_after_docs1.isoformat())
    query8 = f"大阪のことについて、なにか分かることを教えてください。{instruction}"
    answer8, source8 = await rag_with_filter.aquery(query8, param=param8)
    print_test_case("Test Case 8: 'end_time'でフィルタ (前半の登録データのみ)。正解: 大阪の天気について回答できること。これは精度が低いかも", answer8, source8)

    param9 = QueryParam(mode="light",
                        start_time=time_after_docs1.isoformat(), end_time=(time_after_docs1 - timedelta(seconds=10)).isoformat())
    query9 = f"東京について何か情報は登録されましたか？{instruction}"
    answer9, source9 = await rag_with_filter.aquery(query9, param=param9) # スタート時間よりも10秒前で指定している
    print_test_case("Test Case 9: 時間範囲指定 (結果なし)", answer9, source9)

    # --- 複合フィルタリングのテスト ---
    param10 = QueryParam(mode="light",
                         metadata_filter={"category": "geography"}, start_time=time_after_docs1.isoformat())
    query10 = f"日本の地理に関して、最近の話題を探しています。{instruction}"
    answer10, source10 = await rag_with_filter.aquery(query10, param=param10)
    print_test_case("Test Case 10: メタデータと時間の複合フィルタ (成功例。後半の情報を使う)", answer10, source10)


    param10 = QueryParam(mode="light",
                         metadata_filter={"category": "geography"}, end_time=time_after_docs1.isoformat())
    query10 = f"日本の地理に関して、最近の話題を探しています。{instruction}"
    answer10, source10 = await rag_with_filter.aquery(query10, param=param10)
    print_test_case("こっちはエンドタイムでやってみた。(失敗例。metadata_filter: geography とタイムフィルター)", answer10, source10)


    param10 = QueryParam(mode="light",
                                                                     end_time=time_after_docs1.isoformat())
    query10 = f"日本の地理に関して、最近の話題を探しています。{instruction}"
    answer10, source10 = await rag_with_filter.aquery(query10, param=param10)
    print_test_case("こっちはエンドタイムでやってみた。(失敗例。タイムフィルターのみ)", answer10, source10)


    
    param11 = QueryParam(mode="light",
                         metadata_filter={"category": "weather"}, start_time=time_after_docs1.isoformat())
    query11 = f"最近の天気予報について教えてください。{instruction}"
    answer11, source11 = await rag_with_filter.aquery(query11, param=param11)
    print_test_case("Test Case 11: メタデータと時間の複合フィルタ (失敗例)", answer11, source11)

    # --- その他のテスト ---
    query12 = f"フランスの首都について教えてください。{instruction}"
    answer12, source12 = await rag_with_filter.aquery(query12)
    print_test_case("Test Case 12: クエリがどの文書にもマッチしない", answer12, source12)



await run_tests()

------------------------- ポスグレに接続しています -------------------------
------------------------- ポスグレに接続しました！ -------------------------
------------------------- テーブルの存在を確認・作成しています -------------------------
------------------------- テーブルの準備が完了しました！ -------------------------
------------------------- RAGシステムが初期化されました！ -------------------------

[ステップ1] データの登録を開始します...
🚀 AINSERT called with overwrite=True
📥 Input: 2 documents
📥 IDs: ['doc1', 'doc2']
📥 Metadatas: [{'category': 'weather', 'city': 'Tokyo', 'year': 2024}, {'category': 'weather', 'city': 'Osaka', 'year': 2024}]
🔥 OVERWRITE MODE: Deleting existing chunks for 2 documents
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exist

Generating embeddings: 100%|██████████| 1/1 [00:00<00:00, 54.64batch/s]

💾 Upserting chunk 'chunk-7da419c35b...' with metadata: {'city': 'Tokyo', 'year': 2024, 'category': 'weather', 'text_field': '_all'}
🗑️  Deleted text_chunks records for doc_ids: ['doc1', 'doc2']


⠙ Processed 1 chunks, 2 entities(duplicated), 0 relations(duplicated)
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
⚙️  Processing doc 'doc2', status_doc.metadata = {"city": "Osaka", "year": 2024, "category": "weather"}
📦 Created 1 standard chunks for doc 'doc2'
   └─ Chunk 'chunk-4b17ba18b5...' text_field: _all, metadata: {'city': 'Osaka', 'year': 2024, 'category': 'weather', 'text_field': '_all'}


Generating embeddings: 100%|██████████| 1/1 [00:00<00:00, 114.29batch/s]

...

💾 Upserting chunk 'chunk-94cf9b5fb2...' with metadata: {'text_field': '_all'}


⠙ Processed 1 chunks, 2 entities(duplicated), 1 relations(duplicated)
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists


Generating embeddings: 100%|██████████| 1/1 [00:00<00:00, 94.50batch/s]

KG successfully indexed.
データセット2 (doc3, doc4, doc5) を登録しました。

[ステップ2] 検索フィルタリングのテストを実行します...


🎯 Using distance threshold: -1.0
📊 Query returned 60 results
✅ Result 1: id=ent-80dcb02e8c47..., distance=0.4422107058970177
✅ Result 2: id=ent-fed57dfc6f67..., distance=0.39923384377078186
🔎 Total records in LIGHTRAG_VDB_ENTITY: 140
🔎 Raw metadata and distance values:
   - ID: ent-80dcb02e8c47...
     Raw metadata: {'year': 2023, 'country': 'Japan', 'category': 'geography', 'text_field': '_all'}
     Extracted category: geography
     Distance: 0.4422107058970177
   - ID: ent-fed57dfc6f67...
     Raw metadata: {} (no metadata registered)
     Distance: 0.39923384377078186
   - ID: ent-40ebe0b76a4c...
     Raw metadata: {'text_field': '_all'}
     Extracted category: None
     Distance: 0.34531805029319296
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl.py): create_graph already exists
★デバッグ(postgres_impl

### メモ

- Mini モード／Light モードで使用している chunks_vdb.query() すべてに debug=False を明示したので、混乱を招くようなデバッグ表示は出ないようにした。ただし、Naive モードはデバッグ表示が ON になっているのでデバッグ表示の量がちょっと多い。また、Mini モードは一部 ベクトル検索を使っている関係で Naive モードと同じデバッグ表示が表示される。
- デバッグ表示が ON になると以下が表示される。これらの表示が Mini モードや Naive モードで表示されるということ
  - 「Total records in LIGHTRAG_DOC_CHUNKS: X」
  - 「Raw metadata and distance values: ~~~」
    - 「Raw metadata and distance values:」はトップ3が表示される仕様
- ドキュメントIDが重複すると Deleted が出るのは仕様でOK。一回消して再インデックスを作成しているから。
- 大量に出ている表示は正常
  - 「★デバッグ(postgres_impl.py): create_graph already exists」がたくさん出るのは正常。ナレッジグラフがあるかどうか必ずチェックする仕様になっているため。
  - Mini モードの「SELECT * FROM cypher('～～～」 や None も正常
- 次元が混在するとエラーになるので、embedding model は基本的に統一した方が良い
- doc_id はなくてもいいけど、管理しやすくなるのであった方が良さそう
- 「🔎 Raw metadata and distance values:」というデバッグ出力はメタデータのフィルタリング「前」のVDBクエリ結果を表示しており、実際にLLMへ渡す直前のコンテキスト構築ではフィルタ済みのチャンクのみが使われているので、フィルターが適用されていないように見えるが問題なく動作している。
  - メタデータフィルタはVDBクエリ段階では渡さず、チャンク取得後に適用する方針の実装になっています。
  - ローカル文脈構築でのメタデータフィルタ適用。グローバル文脈構築でも同様にチャンク側でフィルタ
  - 一方、デバッグ出力はVDBクエリ（pre-filter）内で行われています（件数・距離・生メタデータの表示）。






In [ ]:
# 以下が色々出るのは仕様なので正常な動作。いっぱい出るのは並列処理しているらしい
# 正常に動いていることを確認したいので、あえて出したままにしている。
# ---------------------------------------------------------
# ★デバッグ(postgres_impl.py): create_graph already exists



# postgres16_age_pgvector_container  | 2025-07-22 04:43:01.467 UTC [1672] ERROR:  graph "my_minirag_graph" already exists
# postgres16_age_pgvector_container  | 2025-07-22 04:43:01.467 UTC [1672] STATEMENT:  select create_graph('my_minirag_graph')



# SELECT * FROM cypher('my_minirag_graph', $$
#                      MATCH path = (start:Entity {node_id: "xe980b1e69cab"})-[*1..2]-(neighbor:Entity)
#                      RETURN [n in nodes(path) | properties(n).node_id] AS path_nodes,
#                             [r in relationships(path) | r] AS path_edges
#                    $$) AS (path_nodes agtype, path_edges agtype)
# None

# 上記のクエリも合っているし Miniモード も正常に機能しているので問題ない。多分表示している箇所は PostgreSQLDB クラスの execute メソッドだと思う。




# INFO:minirag:Using the label default for PostgreSQL as identifier
# INFO:minirag:Connected to PostgreSQL database at postgres:5432/my_database
# ERROR:minirag:PostgreSQL database error: relation "lightrag_doc_full" does not exist
# ERROR:minirag:Failed to check table LIGHTRAG_DOC_FULL in PostgreSQL database
# ERROR:minirag:PostgreSQL database error: relation "lightrag_doc_full" does not exist
# INFO:minirag:Created table LIGHTRAG_DOC_FULL in PostgreSQL database
# ERROR:minirag:PostgreSQL database error: relation "lightrag_doc_chunks" does not exist
# ERROR:minirag:Failed to check table LIGHTRAG_DOC_CHUNKS in PostgreSQL database
# ERROR:minirag:PostgreSQL database error: relation "lightrag_doc_chunks" does not exist
# INFO:minirag:Created table LIGHTRAG_DOC_CHUNKS in PostgreSQL database
# ERROR:minirag:PostgreSQL database error: relation "lightrag_vdb_entity" does not exist
# ERROR:minirag:Failed to check table LIGHTRAG_VDB_ENTITY in PostgreSQL database
# ERROR:minirag:PostgreSQL database error: relation "lightrag_vdb_entity" does not exist
# INFO:minirag:Created table LIGHTRAG_VDB_ENTITY in PostgreSQL database
# ERROR:minirag:PostgreSQL database error: relation "lightrag_vdb_relation" does not exist
# ERROR:minirag:Failed to check table LIGHTRAG_VDB_RELATION in PostgreSQL database
# ERROR:minirag:PostgreSQL database error: relation "lightrag_vdb_relation" does not exist
# INFO:minirag:Created table LIGHTRAG_VDB_RELATION in PostgreSQL database
# ERROR:minirag:PostgreSQL database error: relation "lightrag_llm_cache" does not exist
# ERROR:minirag:Failed to check table LIGHTRAG_LLM_CACHE in PostgreSQL database
# ERROR:minirag:PostgreSQL database error: relation "lightrag_llm_cache" does not exist
# INFO:minirag:Created table LIGHTRAG_LLM_CACHE in PostgreSQL database
# ERROR:minirag:PostgreSQL database error: relation "lightrag_doc_status" does not exist
# ERROR:minirag:Failed to check table LIGHTRAG_DOC_STATUS in PostgreSQL database
# ERROR:minirag:PostgreSQL database error: relation "lightrag_doc_status" does not exist
# INFO:minirag:Created table LIGHTRAG_DOC_STATUS in PostgreSQL database
# INFO:minirag:Finished checking all tables in PostgreSQL database
# INFO:minirag:Logger initialized for working directory: /tmp/minirag_demo

# 上記の表示されている ERROR ログは、一見すると問題のように見えますが、実際には 「テーブルが存在しないことを検知した」 という正常な動作の一部です。check_tables メソッドは、このエラーを意図的に発生させて、テーブルが存在しない場合にのみ作成処理を行うように設計されています。